In [ ]:
import pandas as pd
import numpy as np


CALCULATE GRADIENT


In [ ]:
def gradient(y_true, y_pred):
    """
    Compute the gradient of the loss function for regression.
    
    Parameters:
    y_true (array-like): True target values.
    y_pred (array-like): Predicted target values.
    
    Returns:
    array-like: Gradient values.
    """
    return (y_pred - y_true)

HESSIAN

In [ ]:
def hessian(y_true):
    """
    here the functiion that we use is from numpy i.e.
    np.ones_like() and the function np.ones_like

    Parameters:
    y_true (array-like): True target values.

    
    Returns:
    array-like: Hessian values
    
    """
    return np.ones_like(y_true)

EXACT GREEDY ALGORITH FOR SPLIT FINDING


normally, it is impossible to enumerate all the possible tree structures this is where greedy algorithm comes in. A greedy algorithm starts from a single leaf and iteratively adds branches to the tree is used here

here 

I = I_L U I_R

In [ ]:
import numpy as np

def split_best(X, g, h, reg_lambda=1.0, gamma=0.0):
    """
    Finds the best split (feature and threshold) for a node using XGBoost's Gain metric.
    
    Parameters:
        X (ndarray): Feature matrix of shape (n_samples, n_features)
        g (ndarray): First-order gradients of shape (n_samples,)
        h (ndarray): Second-order gradients (Hessians) of shape (n_samples,)
        reg_lambda (float): L2 regularization parameter (lambda)
        gamma (float): Minimum gain required to make a split
        
    Returns:
        best_gain (float): Maximum gain achieved by the optimal split
        best_feature (int): Index of the feature to split on
        best_threshold (float): Feature value threshold for the split
    """
    n_samples, n_features = X.shape
    
    # Total sum of gradients and hessians for the parent node
    G_P = np.sum(g)
    H_P = np.sum(h)
    
    best_gain = 0.0
    best_feature = None
    best_threshold = None
    
    # Iterate through every feature column
    for feature_idx in range(n_features):
        X_col = X[:, feature_idx]
        
        # Sort samples by current feature values to evaluate splits efficiently
        sort_indices = np.argsort(X_col)
        X_sorted = X_col[sort_indices]
        g_sorted = g[sort_indices]
        h_sorted = h[sort_indices]
        
        # Running cumulative sums for the left child (G_L, H_L)
        G_L = 0.0
        H_L = 0.0
        
        for i in range(n_samples - 1):
            G_L += g_sorted[i]
            H_L += h_sorted[i]
            
            # Skip candidate splits between identical adjacent feature values
            if X_sorted[i] == X_sorted[i + 1]:
                continue
                
            # Right child totals via gradient additivity
            G_R = G_P - G_L
            H_R = H_P - H_L
            
            # Calculate Gain: 0.5 * [Score(L) + Score(R) - Score(P)] - gamma
            score_L = (G_L ** 2) / (H_L + reg_lambda)
            score_R = (G_R ** 2) / (H_R + reg_lambda)
            score_P = (G_P ** 2) / (H_P + reg_lambda)
            
            gain = 0.5 * (score_L + score_R - score_P) - gamma
            
            # Track the best split across all features and thresholds
            if gain > best_gain:
                best_gain = gain
                best_feature = feature_idx
                # Threshold selected as midpoint between adjacent values
                best_threshold = (X_sorted[i] + X_sorted[i + 1]) / 2.0
                
    return best_gain, best_feature, best_threshold
